In [1]:
import numpy as np
import os
import subprocess as subp

class VaspJobs:
    def __init__(self, a0, elem, surf_index, structure='bcc'):
        # Material parameters
        self.a0 = a0  # Lattice constant
        self.elem = elem
        self.structure = structure
        self.dimsize = 12  # Cell dimension along separation direction (units: lattice)

        # VASP calculation setup
        self.kpoints = [14, 14, 1] if surf_index == 100 else [10, 14, 1]
        self.encut = 500
        self.nelm = 360  # Maximum number of SCF cycles
        self.ediff = 1e-6  # Convergence criterion
        self.ncore = 1
        self.ismear = 1
        self.sigma = 0.1

        # SLURM setup
        self.nodes = 2
        self.tasks = 12
        self.mempercpu = '10GB'
        self.exetime = '08:00:00'
        self.partition = 'regularshort'

    def vasp_surface(self, jobscript, separation, surface_index, add_disp=False):
        """
        Generate VASP input for surface separation process.
        :param jobscript: Name of the job script file
        :param separation: Vacuum separation
        :param surface_index: Surface index (e.g., (1, 0, 0) for 100)
        :param add_disp: Whether to add random displacements
        """
        ase_vasp = f"""#Python scripts for submitting VASP jobs.
from ase.build import bulk, surface
from ase.io.extxyz import write_extxyz
from ase.calculators.vasp import Vasp
import numpy as np

# Material parameters
a0 = {self.a0}
dimsize = {self.dimsize}
vac = {separation}
surface_index = {surface_index}  # Pass surface_index into the script

# VASP calculator setup
calc = Vasp(prec='Accurate',
            algo='fast',
            xc="PBE", setups={{'{self.elem}': '_sv'}},
            kpts={self.kpoints},
            encut={self.encut},
            ediff={self.ediff},
            nelm={self.nelm},
            ismear={self.ismear},
            sigma={self.sigma},
            ncore={self.ncore},
            ispin=1,
            isif=0,
            istart=0,
            icharg=2,
            nelmin=5,
            lasph=True,
            lreal=False,
            ldiag='T',
            lwave=False,
            npar=4)

# Create bulk and surface
bulk_structure = bulk('{self.elem}', '{self.structure}', a=a0, cubic=True)
surf = surface(bulk_structure, surface_index, dimsize)
surf.set_pbc([True, True, True])
surf.center(vacuum=vac, axis=2)

# Adjust cell dimensions based on surface orientation
if surface_index == (1, 0, 0):
    surf.set_cell([a0, a0, dimsize*a0 + 2*vac])
elif surface_index == (1, 1, 0):
    surf.set_cell([a0*np.sqrt(2), a0, dimsize*a0*np.sqrt(2)/2 + 2*vac])

# Add random displacements if required
if {add_disp}:
    new_pos = surf.get_positions() + np.random.uniform(low=0.01, high=0.1, size=np.shape(surf.get_positions()))
    surf.set_positions(new_pos)

# Attach calculator and compute
surf.set_calculator(calc)
surf.get_potential_energy()
surf.get_forces()

# Write results to file
write_extxyz('./vasp_results.xyz', surf, append=True)
    """
        with open(jobscript, "w") as f:
            f.write(ase_vasp)

    def submit_jobs(self, jobname, jobscript):
        slurm_script = f"""#!/bin/bash
#SBATCH --job-name={jobname}
#SBATCH --partition={self.partition}
#SBATCH --nodes={self.nodes}
#SBATCH --ntasks-per-node={self.tasks}
#SBATCH --cpus-per-task=1
#SBATCH --mem-per-cpu={self.mempercpu}
#SBATCH --time={self.exetime}
#SBATCH --error=slurm-%j.stderr
#SBATCH --output=slurm-%j.stdout

# Load necessary modules
module load VASP
module load Python

python {jobscript}
"""
        with open("submit", "w") as f:
            f.write(slurm_script)

In [4]:
# Submitting jobs on (100) surface
separation = np.arange(0, 3.5, 0.05)  # Real distance divided by 2, in Angstrom

for vac in separation:
    # Create a directory for each run
    vacs = '{0:.1f}'.format(2 * vac)  # Format the vacuum separation to one decimal place
    direc = f'100_sep_nodisp_{vacs}'  # Directory name
    os.makedirs(direc, exist_ok=True)  # Create the directory if it doesn't exist
    os.chdir(direc)  # Move into the directory

    # Initialize the VaspJobs class
    surf100 = VaspJobs(a0='3.185', elem='W', surf_index=100)  # Example for tungsten (W) with (100) surface

    # Generate the VASP input file
    surf100.vasp_surface("vasp.py", vac, surface_index=(1, 0, 0), add_disp=False)

    # Submit the job using SLURM
    surf100.submit_jobs(direc, "vasp.py")
    subp.run(["sbatch", "submit"])

    # Move back to the parent directory
    os.chdir("../")

Submitted batch job 15181798
Submitted batch job 15181799
Submitted batch job 15181800
Submitted batch job 15181801
Submitted batch job 15181802
Submitted batch job 15181803
Submitted batch job 15181804
Submitted batch job 15181805
Submitted batch job 15181806
Submitted batch job 15181807
Submitted batch job 15181808
Submitted batch job 15181809
Submitted batch job 15181810
Submitted batch job 15181811
Submitted batch job 15181812
Submitted batch job 15181813
Submitted batch job 15181814
Submitted batch job 15181815
Submitted batch job 15181816
Submitted batch job 15181817
Submitted batch job 15181818
Submitted batch job 15181819
Submitted batch job 15181820
Submitted batch job 15181821
Submitted batch job 15181822
Submitted batch job 15181823
Submitted batch job 15181824
Submitted batch job 15181825
Submitted batch job 15181826
Submitted batch job 15181827
Submitted batch job 15181828
Submitted batch job 15181829
Submitted batch job 15181830
Submitted batch job 15181831
Submitted batc

In [5]:
# Submitting jobs on (100) surface with additional displacement
separation = np.arange(0, 3.5, 0.05)  # Real distance divided by 2, in Angstrom

for vac in separation:
    # Create a directory for each run
    vacs = '{0:.1f}'.format(2 * vac)  # Format the vacuum separation to one decimal place
    direc = f'100_sep_disp_{vacs}'  # Directory name
    os.makedirs(direc, exist_ok=True)  # Create the directory if it doesn't exist
    os.chdir(direc)  # Move into the directory

    # Initialize the VaspJobs class
    surf100 = VaspJobs(a0='3.185', elem='W', surf_index=100)  # Example for tungsten (W) with (100) surface

    # Generate the VASP input file
    surf100.vasp_surface("vasp.py", vac, surface_index=(1, 0, 0), add_disp=True)

    # Submit the job using SLURM
    surf100.submit_jobs(direc, "vasp.py")
    subp.run(["sbatch", "submit"])

    # Move back to the parent directory
    os.chdir("../")

Submitted batch job 15181868
Submitted batch job 15181869
Submitted batch job 15181870
Submitted batch job 15181871
Submitted batch job 15181872
Submitted batch job 15181873
Submitted batch job 15181874
Submitted batch job 15181875
Submitted batch job 15181876
Submitted batch job 15181877
Submitted batch job 15181878
Submitted batch job 15181879
Submitted batch job 15181880
Submitted batch job 15181881
Submitted batch job 15181882
Submitted batch job 15181883
Submitted batch job 15181884
Submitted batch job 15181885
Submitted batch job 15181886
Submitted batch job 15181887
Submitted batch job 15181888
Submitted batch job 15181889
Submitted batch job 15181890
Submitted batch job 15181891
Submitted batch job 15181892
Submitted batch job 15181893
Submitted batch job 15181894
Submitted batch job 15181895
Submitted batch job 15181896
Submitted batch job 15181897
Submitted batch job 15181898
Submitted batch job 15181899
Submitted batch job 15181900
Submitted batch job 15181901
Submitted batc

In [8]:
# Submitting jobs on (110) surface
separation = np.arange(0, 3.5, 0.05)  # Real distance divided by 2, in Angstrom

for vac in separation:
    # Create a directory for each run
    vacs = '{0:.1f}'.format(2 * vac)  # Format the vacuum separation to one decimal place
    direc = f'110_sep_nodisp_{vacs}'  # Directory name
    os.makedirs(direc, exist_ok=True)  # Create the directory if it doesn't exist
    os.chdir(direc)  # Move into the directory

    # Initialize the VaspJobs class
    surf110 = VaspJobs(a0='3.185', elem='W', surf_index=110)  # Example for tungsten (W) with (110) surface

    # Generate the VASP input file
    surf110.vasp_surface("vasp.py", vac, surface_index=(1, 1, 0), add_disp=False)

    # Submit the job using SLURM
    surf110.submit_jobs(direc, "vasp.py")
    subp.run(["sbatch", "submit"])

    # Move back to the parent directory
    os.chdir("../")

Submitted batch job 15182085
Submitted batch job 15182086
Submitted batch job 15182087
Submitted batch job 15182088
Submitted batch job 15182089
Submitted batch job 15182090
Submitted batch job 15182091
Submitted batch job 15182092
Submitted batch job 15182093
Submitted batch job 15182094
Submitted batch job 15182095
Submitted batch job 15182096
Submitted batch job 15182097
Submitted batch job 15182098
Submitted batch job 15182099
Submitted batch job 15182100
Submitted batch job 15182101
Submitted batch job 15182102
Submitted batch job 15182103
Submitted batch job 15182104
Submitted batch job 15182105
Submitted batch job 15182106
Submitted batch job 15182107
Submitted batch job 15182108
Submitted batch job 15182109
Submitted batch job 15182110
Submitted batch job 15182111
Submitted batch job 15182112
Submitted batch job 15182113
Submitted batch job 15182114
Submitted batch job 15182115
Submitted batch job 15182116
Submitted batch job 15182117
Submitted batch job 15182118
Submitted batc

In [9]:
# Submitting jobs on (110) surface
separation = np.arange(0, 3.5, 0.05)  # Real distance divided by 2, in Angstrom

for vac in separation:
    # Create a directory for each run
    vacs = '{0:.1f}'.format(2 * vac)  # Format the vacuum separation to one decimal place
    direc = f'110_sep_disp_{vacs}'  # Directory name
    os.makedirs(direc, exist_ok=True)  # Create the directory if it doesn't exist
    os.chdir(direc)  # Move into the directory

    # Initialize the VaspJobs class
    surf110 = VaspJobs(a0='3.185', elem='W', surf_index=110)  # Example for tungsten (W) with (110) surface

    # Generate the VASP input file
    surf110.vasp_surface("vasp.py", vac, surface_index=(1, 1, 0), add_disp=True)

    # Submit the job using SLURM
    surf110.submit_jobs(direc, "vasp.py")
    subp.run(["sbatch", "submit"])

    # Move back to the parent directory
    os.chdir("../")

Submitted batch job 15182155
Submitted batch job 15182156
Submitted batch job 15182157
Submitted batch job 15182158
Submitted batch job 15182159
Submitted batch job 15182160
Submitted batch job 15182161
Submitted batch job 15182162
Submitted batch job 15182163
Submitted batch job 15182164
Submitted batch job 15182165
Submitted batch job 15182166
Submitted batch job 15182167
Submitted batch job 15182168
Submitted batch job 15182169
Submitted batch job 15182170
Submitted batch job 15182171
Submitted batch job 15182172
Submitted batch job 15182173
Submitted batch job 15182174
Submitted batch job 15182175
Submitted batch job 15182176
Submitted batch job 15182177
Submitted batch job 15182178
Submitted batch job 15182179
Submitted batch job 15182180
Submitted batch job 15182181
Submitted batch job 15182182
Submitted batch job 15182183
Submitted batch job 15182184
Submitted batch job 15182185
Submitted batch job 15182186
Submitted batch job 15182187
Submitted batch job 15182188
Submitted batc